In [ ]:
# | default_exp preprocessing.ocr.layout_bundle

# Layout-aware OCR bundle ingestion

Canonicalize schema-v2 OCR bundles without losing exact region evidence or provenance.

In [ ]:
# | export
import hashlib
import html
import json
import re
import unicodedata
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from difflib import SequenceMatcher
from html.parser import HTMLParser
from pathlib import Path
from typing import Any, Iterable, Iterator, Literal, Mapping, Sequence

try:
    from ribosome.preprocessing.ocr.audit_repair import audit_layout_ocr_file
except ModuleNotFoundError:  # The exported adapter remains usable in a minimal wheel.
    audit_layout_ocr_file = None


LAYOUT_PIPELINE_VERSION = "layout-ocr-rag-v1"
SUPPORTED_LAYOUT_SCHEMA_VERSIONS = frozenset({2})

_REGION_MARKER_RE = re.compile(
    r"^<!--\s*layout-region\s+"
    r"page=(?P<page>\d+)\s+"
    r"index=(?P<index>\d+)\s+"
    r"label=(?P<label>\S+)\s+"
    r"bbox=(?P<x1>-?\d+),(?P<y1>-?\d+),(?P<x2>-?\d+),(?P<y2>-?\d+)\s+"
    r"status=(?P<status>\S+)\s*-->\s*$",
    re.MULTILINE,
)
_PAGE_MARKER_RE = re.compile(r"^<!--\s*Page\s+(?P<page>\d+)\s*-->\s*$", re.MULTILINE)
_NUMBERED_HEADING_RE = re.compile(
    r"^\s*(?P<number>\d+(?:\s*\.\s*\d+)*)\s+(?P<title>\S.*)$"
)
_IDENTIFIER_RE = re.compile(
    r"(?<![A-Za-z0-9_])(?:[A-Z][A-Z0-9_]*(?:\[\])?(?:\.[A-Za-z]+)?)(?![A-Za-z0-9_])"
)
_MARKDOWN_HEADING_RE = re.compile(r"^\s*#{1,6}\s+", re.MULTILINE)
_WHITESPACE_RE = re.compile(r"\s+")

COMMON_IDENTIFIER_ALIASES: Mapping[str, str] = {
    "SIRLEN": "STRLEN",
    "STREVERSE": "STRREVERSE",
    "OUT T": "OUT_T",
    "OUT_T": "OUT_T",
    "I0": "IO",
    "I/0": "I/O",
}

MULTIWORD_INSTRUCTION_ALIASES: Mapping[str, str] = {
    "START BG": "START BG",
    "STOP BG": "STOP BG",
    "SKIP CONDITION": "SKIP CONDITION",
    "TIMER START": "TIMER START",
    "TIMER STOP": "TIMER STOP",
    "USER OFFSET CONDITION": "USER_OFFSET_CONDITION",
    "USER_OFFSET_CONDITION": "USER_OFFSET_CONDITION",
    "TOOL OFFSET CONDITION": "TOOL_OFFSET_CONDITION",
    "TOOL_OFFSET_CONDITION": "TOOL_OFFSET_CONDITION",
}


def _sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path: str | Path) -> str:
    """Return the SHA-256 digest for an immutable source artifact."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _compact_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def _clean_text(value: object) -> str:
    return str(value or "").replace("\r\n", "\n").replace("\r", "\n").strip()


def _plain_markdown(value: str) -> str:
    value = _MARKDOWN_HEADING_RE.sub("", value)
    value = re.sub(r"!\[[^\]]*\]\(<[^>]+>\)", "", value)
    value = re.sub(r"^>\s*\[!WARNING\].*$", "", value, flags=re.MULTILINE)
    return value.strip()


def _normalized_compare_text(value: str) -> str:
    normalized = unicodedata.normalize("NFKC", html.unescape(value)).casefold()
    return "".join(character for character in normalized if not character.isspace())


def _syntax_signature(value: str) -> tuple[str, ...]:
    normalized = unicodedata.normalize("NFKC", html.unescape(value))
    normalized = normalized.upper()
    for raw_term, canonical_term in COMMON_IDENTIFIER_ALIASES.items():
        normalized = normalized.replace(raw_term, canonical_term)
    tokens = re.findall(
        r"(?:!=|==|<=|>=|:=|&&|\|\||[=<>+*/%])|"
        r"(?:[A-Z][A-Z0-9_]*(?:\[[^\]]*\])?(?:\.[A-Za-z]+)?)",
        normalized,
    )
    return tuple(tokens)


def _operator_signature(value: str) -> tuple[str, ...]:
    """Return operators whose substitution could change executable meaning."""
    normalized = unicodedata.normalize("NFKC", html.unescape(value))
    return tuple(dict.fromkeys(re.findall(r"!=|==|<=|>=|:=|&&|\|\||[=<>+*%]", normalized)))


def _canonical_heading_title(value: str) -> str:
    """Normalize known OCR mnemonic confusions without losing descriptive suffixes."""
    cleaned = _WHITESPACE_RE.sub(" ", unicodedata.normalize("NFKC", value)).strip()
    upper = cleaned.upper()
    for raw, canonical in sorted(MULTIWORD_INSTRUCTION_ALIASES.items(), key=lambda item: -len(item[0])):
        if upper == raw or upper.startswith(f"{raw} "):
            return canonical + cleaned[len(raw) :]
    for raw, canonical in sorted(COMMON_IDENTIFIER_ALIASES.items(), key=lambda item: -len(item[0])):
        if upper == raw or upper.startswith(f"{raw} "):
            return canonical + cleaned[len(raw) :]
    return cleaned


def _stable_semantic_layout_projection(payload: Mapping[str, Any]) -> dict[str, Any]:
    """Project a sidecar to path- and request-metadata-independent evidence."""
    pages: list[dict[str, Any]] = []
    for page in payload.get("pages") or []:
        if not isinstance(page, Mapping):
            continue
        regions: list[dict[str, Any]] = []
        for region in page.get("regions") or []:
            if not isinstance(region, Mapping):
                continue
            asset = region.get("asset")
            regions.append(
                {
                    "index": region.get("index"),
                    "label": region.get("label"),
                    "score": region.get("score"),
                    "bbox": region.get("bbox"),
                    "task_type": region.get("task_type"),
                    "status": region.get("status"),
                    "asset": Path(asset).name if isinstance(asset, str) else None,
                    "raw_content": region.get("raw_content"),
                    "content": region.get("content"),
                    "error": region.get("error"),
                    "recovery": region.get("recovery"),
                    "finish_reason": region.get("finish_reason"),
                }
            )
        pages.append(
            {
                "page_number": page.get("page_number"),
                "width": page.get("width"),
                "height": page.get("height"),
                "status": page.get("status"),
                "regions": regions,
            }
        )
    return {
        "schema_version": payload.get("schema_version"),
        "status": payload.get("status"),
        "provider": payload.get("provider"),
        "model": payload.get("model"),
        "layout_model": payload.get("layout_model"),
        "dpi": payload.get("dpi"),
        "pages": pages,
    }

In [ ]:
# | export
ValidationSeverity = Literal["warning", "error"]
QualitySeverity = Literal["info", "warning", "error"]


@dataclass(frozen=True)
class BoundingBox:
    """A pixel-space top-left-origin layout bounding box."""

    x1: int
    y1: int
    x2: int
    y2: int

    @classmethod
    def from_value(cls, value: object) -> "BoundingBox | None":
        if not isinstance(value, Sequence) or isinstance(value, (str, bytes)):
            return None
        try:
            coordinates = tuple(int(coordinate) for coordinate in value)
        except (TypeError, ValueError):
            return None
        if len(coordinates) != 4:
            return None
        return cls(*coordinates)

    def as_tuple(self) -> tuple[int, int, int, int]:
        return self.x1, self.y1, self.x2, self.y2

    def normalized(self, width: int, height: int) -> tuple[float, float, float, float]:
        if width <= 0 or height <= 0:
            raise ValueError("page dimensions must be positive")
        return (
            self.x1 / width,
            self.y1 / height,
            self.x2 / width,
            self.y2 / height,
        )

    def contains(self, other: "BoundingBox", tolerance: int = 0) -> bool:
        return (
            self.x1 - tolerance <= other.x1
            and self.y1 - tolerance <= other.y1
            and self.x2 + tolerance >= other.x2
            and self.y2 + tolerance >= other.y2
        )


@dataclass(frozen=True)
class BundleValidationIssue:
    code: str
    message: str
    severity: ValidationSeverity = "error"
    page_number: int | None = None
    region_index: int | None = None


class LayoutBundleValidationError(ValueError):
    """Raised when bundle artifacts cannot be joined without losing provenance."""

    def __init__(self, issues: Sequence[BundleValidationIssue]):
        self.issues = tuple(issues)
        details = "; ".join(issue.message for issue in self.issues[:8])
        if len(self.issues) > 8:
            details += f"; and {len(self.issues) - 8} more"
        super().__init__(details or "invalid layout bundle")


@dataclass(frozen=True)
class QualityFlag:
    code: str
    reason: str
    severity: QualitySeverity = "warning"
    resolved: bool = False
    resolution: str | None = None


@dataclass(frozen=True)
class MarkdownRegion:
    page_number: int
    region_index: int
    label: str
    bbox: BoundingBox
    status: str
    payload: str
    marker: str

    @property
    def key(self) -> tuple[int, int]:
        return self.page_number, self.region_index


@dataclass(frozen=True)
class LayoutBundlePaths:
    layout_path: Path
    markdown_path: Path
    pdf_path: Path | None
    assets_path: Path

    @property
    def stem(self) -> str:
        name = self.layout_path.name
        if name.endswith(".partial.layout.json"):
            return name[: -len(".partial.layout.json")]
        return name[: -len(".layout.json")] if name.endswith(".layout.json") else self.layout_path.stem

    def resolve_asset(self, asset: object, *, prefix: object = None) -> Path | None:
        if not isinstance(asset, str) or not asset.strip():
            return None
        path = Path(asset)
        if path.is_absolute():
            return path.resolve()
        root = self.layout_path.parent
        if isinstance(prefix, str) and prefix.strip():
            prefix_path = Path(prefix)
            root = prefix_path if prefix_path.is_absolute() else root / prefix_path
        return (root / path).resolve()

    @classmethod
    def from_layout(
        cls,
        layout_path: str | Path,
        *,
        markdown_path: str | Path | None = None,
        pdf_path: str | Path | None = None,
        pdf_roots: Sequence[str | Path] = (),
    ) -> "LayoutBundlePaths":
        layout = Path(layout_path).expanduser().resolve()
        is_partial = layout.name.endswith(".partial.layout.json")
        stem = (
            layout.name[: -len(".partial.layout.json")]
            if is_partial
            else layout.name[: -len(".layout.json")]
            if layout.name.endswith(".layout.json")
            else layout.stem
        )
        markdown = (
            Path(markdown_path).expanduser().resolve()
            if markdown_path is not None
            else layout.with_name(f"{stem}.partial.md" if is_partial else f"{stem}.md")
        )
        resolved_pdf = Path(pdf_path).expanduser().resolve() if pdf_path is not None else None
        if resolved_pdf is None:
            sibling = layout.with_name(f"{stem}.pdf")
            if sibling.exists():
                resolved_pdf = sibling
        if resolved_pdf is None and pdf_roots:
            candidates: list[Path] = []
            for root in pdf_roots:
                candidates.extend(Path(root).expanduser().resolve().rglob(f"{stem}.pdf"))
            candidates = sorted({candidate.resolve() for candidate in candidates})
            if len(candidates) == 1:
                resolved_pdf = candidates[0]
            elif len(candidates) > 1:
                raise ValueError(f"multiple PDFs match layout stem {stem!r}: {candidates}")
        return cls(
            layout_path=layout,
            markdown_path=markdown,
            pdf_path=resolved_pdf,
            assets_path=layout.with_name(f"{stem}.assets"),
        )


@dataclass
class LayoutRegion:
    document_id: str
    page_number: int
    page_width: int
    page_height: int
    region_index: int
    label: str
    detector_score: float | None
    bbox: BoundingBox
    task_type: str
    ocr_status: str
    raw_content: str
    ocr_content: str
    raw_markdown: str
    asset_path: Path | None
    provenance: dict[str, Any]
    canonical_text: str = ""
    native_text: str = ""
    text_source: str = "ocr"
    quality_flags: tuple[QualityFlag, ...] = ()
    quarantined: bool = False
    previous_region_id: str | None = None
    next_region_id: str | None = None
    section_number: str | None = None
    section_path: str = ""
    parent_id: str = ""
    instruction_code: str | None = None
    is_boilerplate: bool = False
    is_toc: bool = False
    nested_parent_region_id: str | None = None

    @property
    def key(self) -> tuple[int, int]:
        return self.page_number, self.region_index

    @property
    def region_id(self) -> str:
        return f"{self.document_id}:p{self.page_number:04d}:r{self.region_index:04d}"

    @property
    def bbox_normalized(self) -> tuple[float, float, float, float]:
        return self.bbox.normalized(self.page_width, self.page_height)

    @property
    def indexable(self) -> bool:
        return not (self.quarantined or self.is_boilerplate)


@dataclass
class LayoutPage:
    page_number: int
    width: int
    height: int
    status: str
    error: str | None
    regions: tuple[LayoutRegion, ...]


@dataclass
class LayoutSection:
    parent_id: str
    document_id: str
    number: str | None
    title: str
    level: int
    section_path: str
    parent_section_id: str | None
    page_start: int
    page_end: int
    region_ids: list[str] = field(default_factory=list)
    summary: str = ""
    instruction_code: str | None = None
    is_toc: bool = False


@dataclass
class LayoutDocument:
    document_id: str
    document_title: str
    document_code: str
    revision: str
    schema_version: int
    status: str
    paths: LayoutBundlePaths
    artifact_hashes: dict[str, str]
    provider: str
    ocr_model: str
    layout_model: str
    dpi: int | None
    pages: tuple[LayoutPage, ...]
    raw_layout: dict[str, Any]
    pipeline_version: str = LAYOUT_PIPELINE_VERSION
    sections: tuple[LayoutSection, ...] = ()
    validation_issues: tuple[BundleValidationIssue, ...] = ()

    @property
    def regions(self) -> tuple[LayoutRegion, ...]:
        return tuple(region for page in self.pages for region in page.regions)

    @property
    def region_map(self) -> dict[str, LayoutRegion]:
        return {region.region_id: region for region in self.regions}


@dataclass
class LayoutRetrievalRecord:
    chunk_id: str
    parent_id: str
    document_id: str
    document_title: str
    document_code: str
    revision: str
    section_path: str
    instruction_code: str | None
    content_type: str
    retrieval_text: str
    exact_text: str
    raw_markdown: str
    raw_ocr: str
    canonical_texts: tuple[str, ...]
    cleaned_ocr_texts: tuple[str, ...]
    native_texts: tuple[str, ...]
    detector_scores: tuple[float | None, ...]
    page_start: int
    page_end: int
    region_ids: tuple[str, ...]
    bboxes: tuple[tuple[int, int, int, int], ...]
    bboxes_normalized: tuple[tuple[float, float, float, float], ...]
    region_labels: tuple[str, ...]
    ocr_statuses: tuple[str, ...]
    quality_flags: tuple[str, ...]
    quality_reasons: tuple[str, ...]
    region_quality_flags: tuple[tuple[dict[str, Any], ...], ...]
    text_sources: tuple[str, ...]
    asset_paths: tuple[str, ...]
    region_asset_paths: tuple[str | None, ...]
    source_pdf_uri: str | None
    source_markdown_uri: str
    source_layout_uri: str
    source_hashes: tuple[tuple[str, str], ...]
    content_sha256: str
    pipeline_version: str
    aliases: tuple[str, ...] = ()
    indexable: bool = True
    reading_order: int = 0

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class LayoutIngestionResult:
    document: LayoutDocument
    sections: tuple[LayoutSection, ...]
    records: tuple[LayoutRetrievalRecord, ...]

    @property
    def indexable_records(self) -> tuple[LayoutRetrievalRecord, ...]:
        return tuple(record for record in self.records if record.indexable)

In [ ]:
# | export
def parse_markdown_regions(markdown: str) -> tuple[MarkdownRegion, ...]:
    """Parse marker-delimited Markdown payloads without interpreting their content."""
    matches = list(_REGION_MARKER_RE.finditer(markdown))
    parsed: list[MarkdownRegion] = []
    for position, match in enumerate(matches):
        end = matches[position + 1].start() if position + 1 < len(matches) else len(markdown)
        payload = _PAGE_MARKER_RE.sub("", markdown[match.end() : end]).strip()
        parsed.append(
            MarkdownRegion(
                page_number=int(match.group("page")),
                region_index=int(match.group("index")),
                label=match.group("label"),
                bbox=BoundingBox(
                    int(match.group("x1")),
                    int(match.group("y1")),
                    int(match.group("x2")),
                    int(match.group("y2")),
                ),
                status=match.group("status"),
                payload=payload,
                marker=match.group(0).strip(),
            )
        )
    return tuple(parsed)


class LayoutBundleValidator:
    """Validate schema-v2 geometry, assets, ordering, and Markdown joins."""

    bundle_statuses = frozenset({"pending", "processing", "completed", "processed", "partial", "failed"})
    page_statuses = frozenset({"pending", "processing", "completed", "processed", "partial", "failed"})
    region_statuses = frozenset(
        {"pending", "processing", "completed", "partial", "failed", "preserved", "recovered"}
    )
    task_types = frozenset({"text", "table", "figure", "formula"})

    def validate(
        self,
        paths: LayoutBundlePaths,
        *,
        strict: bool = True,
    ) -> tuple[BundleValidationIssue, ...]:
        issues: list[BundleValidationIssue] = []
        if not paths.layout_path.is_file():
            issues.append(BundleValidationIssue("missing_layout", f"layout sidecar does not exist: {paths.layout_path}"))
        if not paths.markdown_path.is_file():
            issues.append(BundleValidationIssue("missing_markdown", f"Markdown artifact does not exist: {paths.markdown_path}"))
        if issues:
            if strict:
                raise LayoutBundleValidationError(issues)
            return tuple(issues)

        try:
            payload = json.loads(paths.layout_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError) as error:
            issues.append(BundleValidationIssue("invalid_layout_json", f"cannot parse {paths.layout_path}: {error}"))
            if strict:
                raise LayoutBundleValidationError(issues)
            return tuple(issues)
        if not isinstance(payload, dict):
            issues.append(BundleValidationIssue("invalid_layout_root", "layout JSON root must be an object"))
            if strict:
                raise LayoutBundleValidationError(issues)
            return tuple(issues)

        schema_version = payload.get("schema_version")
        if schema_version not in SUPPORTED_LAYOUT_SCHEMA_VERSIONS:
            issues.append(
                BundleValidationIssue(
                    "unsupported_schema",
                    f"unsupported layout schema_version {schema_version!r}; expected one of {sorted(SUPPORTED_LAYOUT_SCHEMA_VERSIONS)}",
                )
            )
        bundle_status = str(payload.get("status") or "")
        if bundle_status not in self.bundle_statuses:
            issues.append(BundleValidationIssue("unknown_bundle_status", f"unknown bundle status {bundle_status!r}"))

        pages = payload.get("pages")
        asset_prefix = payload.get("asset_prefix")
        bundle_root = paths.layout_path.parent.resolve()
        asset_root = bundle_root
        if isinstance(asset_prefix, str) and asset_prefix.strip():
            prefix_path = Path(asset_prefix)
            asset_root = (prefix_path if prefix_path.is_absolute() else bundle_root / prefix_path).resolve()
            try:
                asset_root.relative_to(bundle_root)
            except ValueError:
                issues.append(
                    BundleValidationIssue(
                        "unsafe_asset_prefix",
                        f"asset_prefix escapes the bundle directory: {asset_prefix!r}",
                    )
                )
        if not isinstance(pages, list):
            issues.append(BundleValidationIssue("invalid_pages", "layout pages must be a list"))
            pages = []
        expected_pages = payload.get("pages_total")
        if isinstance(expected_pages, int) and expected_pages != len(pages):
            issues.append(
                BundleValidationIssue(
                    "page_count_mismatch",
                    f"pages_total={expected_pages} but sidecar contains {len(pages)} pages",
                )
            )

        json_regions: dict[tuple[int, int], tuple[str, BoundingBox, str]] = {}
        previous_page_number = 0
        observed_page_numbers: list[int] = []
        for raw_page in pages:
            if not isinstance(raw_page, dict):
                issues.append(BundleValidationIssue("invalid_page", "each page must be an object"))
                continue
            page_number = raw_page.get("page_number")
            if not isinstance(page_number, int) or page_number <= 0:
                issues.append(BundleValidationIssue("invalid_page_number", f"invalid page_number {page_number!r}"))
                continue
            if page_number <= previous_page_number:
                issues.append(BundleValidationIssue("page_order", f"page {page_number} is duplicated or out of order", page_number=page_number))
            previous_page_number = page_number
            observed_page_numbers.append(page_number)
            width, height = raw_page.get("width"), raw_page.get("height")
            if not isinstance(width, int) or not isinstance(height, int) or width <= 0 or height <= 0:
                issues.append(BundleValidationIssue("invalid_page_dimensions", f"page {page_number} has invalid dimensions {width!r}x{height!r}", page_number=page_number))
                width = height = 0
            page_status = str(raw_page.get("status") or "")
            if page_status not in self.page_statuses:
                issues.append(BundleValidationIssue("unknown_page_status", f"page {page_number} has unknown status {page_status!r}", page_number=page_number))
            raw_regions = raw_page.get("regions")
            if not isinstance(raw_regions, list):
                issues.append(BundleValidationIssue("invalid_regions", f"page {page_number} regions must be a list", page_number=page_number))
                continue
            previous_index = 0
            observed_indices: list[int] = []
            for raw_region in raw_regions:
                if not isinstance(raw_region, dict):
                    issues.append(BundleValidationIssue("invalid_region", f"page {page_number} contains a non-object region", page_number=page_number))
                    continue
                index = raw_region.get("index")
                if not isinstance(index, int) or index <= 0:
                    issues.append(BundleValidationIssue("invalid_region_index", f"page {page_number} has invalid region index {index!r}", page_number=page_number))
                    continue
                if index <= previous_index:
                    issues.append(BundleValidationIssue("region_order", f"page {page_number} region {index} is duplicated or out of order", page_number=page_number, region_index=index))
                previous_index = index
                observed_indices.append(index)
                key = (page_number, index)
                if key in json_regions:
                    issues.append(BundleValidationIssue("duplicate_region", f"duplicate JSON region p{page_number}r{index}", page_number=page_number, region_index=index))
                    continue
                bbox = BoundingBox.from_value(raw_region.get("bbox"))
                if bbox is None or not (0 <= bbox.x1 < bbox.x2 <= width and 0 <= bbox.y1 < bbox.y2 <= height):
                    issues.append(BundleValidationIssue("invalid_bbox", f"page {page_number} region {index} has invalid bbox {raw_region.get('bbox')!r} for {width}x{height}", page_number=page_number, region_index=index))
                    bbox = bbox or BoundingBox(0, 0, 0, 0)
                status = str(raw_region.get("status") or "")
                if status not in self.region_statuses:
                    issues.append(BundleValidationIssue("unknown_region_status", f"page {page_number} region {index} has unknown status {status!r}", page_number=page_number, region_index=index))
                task_type = str(raw_region.get("task_type") or "")
                if task_type not in self.task_types:
                    issues.append(BundleValidationIssue("unknown_task_type", f"page {page_number} region {index} has unknown task type {task_type!r}", page_number=page_number, region_index=index))
                asset_value = raw_region.get("asset")
                asset_path = paths.resolve_asset(asset_value, prefix=asset_prefix)
                if asset_path is not None:
                    try:
                        asset_path.relative_to(bundle_root)
                    except ValueError:
                        issues.append(BundleValidationIssue("unsafe_asset_path", f"page {page_number} region {index} asset escapes its bundle root: {asset_value!r}", page_number=page_number, region_index=index))
                if asset_path is not None and not asset_path.is_file():
                    issues.append(BundleValidationIssue("missing_asset", f"page {page_number} region {index} asset does not exist: {asset_path}", page_number=page_number, region_index=index))
                json_regions[key] = (str(raw_region.get("label") or ""), bbox, status)
            if observed_indices != list(range(1, len(observed_indices) + 1)):
                issues.append(BundleValidationIssue("region_index_gap", f"page {page_number} region indices are not contiguous from 1", page_number=page_number))

        if observed_page_numbers != list(range(1, len(observed_page_numbers) + 1)):
            issues.append(BundleValidationIssue("page_number_gap", "page numbers are not contiguous from 1"))

        markdown = paths.markdown_path.read_text(encoding="utf-8")
        markdown_page_numbers = [int(match.group("page")) for match in _PAGE_MARKER_RE.finditer(markdown)]
        if markdown_page_numbers != observed_page_numbers:
            issues.append(BundleValidationIssue("markdown_page_markers", "Markdown page delimiters do not match ordered JSON pages"))
        parsed_markers = parse_markdown_regions(markdown)
        marker_map: dict[tuple[int, int], MarkdownRegion] = {}
        for marker in parsed_markers:
            if marker.key in marker_map:
                issues.append(BundleValidationIssue("duplicate_markdown_marker", f"duplicate Markdown marker p{marker.page_number}r{marker.region_index}", page_number=marker.page_number, region_index=marker.region_index))
            marker_map[marker.key] = marker
        for key, (label, bbox, status) in json_regions.items():
            marker = marker_map.get(key)
            if marker is None:
                issues.append(BundleValidationIssue("missing_markdown_marker", f"JSON region p{key[0]}r{key[1]} has no Markdown marker", page_number=key[0], region_index=key[1]))
                continue
            if (marker.label, marker.bbox, marker.status) != (label, bbox, status):
                issues.append(BundleValidationIssue("marker_mismatch", f"Markdown marker p{key[0]}r{key[1]} does not match JSON label/bbox/status", page_number=key[0], region_index=key[1]))
        for key in marker_map.keys() - json_regions.keys():
            issues.append(BundleValidationIssue("orphan_markdown_marker", f"Markdown marker p{key[0]}r{key[1]} has no JSON region", page_number=key[0], region_index=key[1]))

        if paths.pdf_path is not None:
            if not paths.pdf_path.is_file():
                issues.append(BundleValidationIssue("missing_pdf", f"configured source PDF does not exist: {paths.pdf_path}"))
            else:
                signature = payload.get("source_signature")
                if isinstance(signature, Mapping) and isinstance(signature.get("size"), int):
                    actual_size = paths.pdf_path.stat().st_size
                    if actual_size != signature["size"]:
                        issues.append(BundleValidationIssue("pdf_size_mismatch", f"source PDF size {actual_size} does not match recorded size {signature['size']}"))
                try:
                    import fitz

                    with fitz.open(paths.pdf_path) as pdf:
                        if pages and len(pdf) != len(pages):
                            issues.append(BundleValidationIssue("pdf_page_count_mismatch", f"source PDF has {len(pdf)} pages but layout has {len(pages)}"))
                except (ImportError, OSError, RuntimeError) as error:
                    issues.append(BundleValidationIssue("pdf_unreadable", f"cannot inspect source PDF: {error}"))

        if strict and any(issue.severity == "error" for issue in issues):
            raise LayoutBundleValidationError(issues)
        return tuple(issues)


def pair_layout_bundles(
    layout_root: str | Path,
    *,
    markdown_root: str | Path | None = None,
    pdf_roots: Sequence[str | Path] = (),
    include_partial: bool = False,
) -> tuple[LayoutBundlePaths, ...]:
    """Pair sidecars by relative stem, optionally across separate PDF roots."""
    layout_root_path = Path(layout_root).expanduser().resolve()
    markdown_root_path = Path(markdown_root).expanduser().resolve() if markdown_root is not None else layout_root_path
    bundles: list[LayoutBundlePaths] = []
    for layout_path in sorted(layout_root_path.rglob("*.layout.json")):
        is_partial = layout_path.name.endswith(".partial.layout.json")
        if not include_partial and is_partial:
            continue
        relative = layout_path.relative_to(layout_root_path)
        stem = layout_path.name[: -len(".partial.layout.json")] if is_partial else layout_path.name[: -len(".layout.json")]
        markdown = markdown_root_path / relative.parent / (f"{stem}.partial.md" if is_partial else f"{stem}.md")
        pdf_candidates = [Path(root).expanduser().resolve() / relative.parent / f"{stem}.pdf" for root in pdf_roots]
        existing_pdfs = [candidate for candidate in pdf_candidates if candidate.is_file()]
        if len(existing_pdfs) > 1:
            raise ValueError(f"multiple source PDFs match {relative}: {existing_pdfs}")
        bundles.append(
            LayoutBundlePaths.from_layout(
                layout_path,
                markdown_path=markdown,
                pdf_path=existing_pdfs[0] if existing_pdfs else None,
            )
        )
    return tuple(bundles)

In [ ]:
# | export
class OCRQualityGate:
    """Map OCR audit findings to explicit flags and production quarantine state."""

    severe_reason_markers = (
        "failed",
        "no usable ocr content",
        "token limit",
        "degenerate ocr output",
        "evaluator commentary",
        "repeated image placeholders",
        "control token",
        "malformed html",
    )
    native_recoverable_codes = frozenset(
        {
            "failed",
            "incomplete",
            "suspicious_completed",
            "ocr_error",
            "empty_content",
            "empty_canonical_text",
            "output_truncated",
            "malformed_html_table",
            "ocr_control_token",
        }
    )

    def audit_flags(self, sidecar_path: str | Path) -> dict[tuple[int, int], tuple[QualityFlag, ...]]:
        if audit_layout_ocr_file is None:
            return self._fallback_audit_flags(sidecar_path)
        report = audit_layout_ocr_file(sidecar_path)
        flags: dict[tuple[int, int], tuple[QualityFlag, ...]] = {}
        for issue in report.region_issues:
            severity: QualitySeverity = "error" if issue.issue_kind in {"failed", "suspicious_completed"} else "warning"
            region_flags = [QualityFlag(issue.issue_kind, reason, severity) for reason in issue.reasons]
            if issue.error:
                region_flags.append(QualityFlag("ocr_error", issue.error, "error"))
            flags[(issue.page_number, issue.region_index)] = tuple(region_flags)
        return flags

    @staticmethod
    def _fallback_audit_flags(sidecar_path: str | Path) -> dict[tuple[int, int], tuple[QualityFlag, ...]]:
        payload = json.loads(Path(sidecar_path).read_text(encoding="utf-8"))
        findings: dict[tuple[int, int], tuple[QualityFlag, ...]] = {}
        evaluator_markers = ("ground truth image", "according to rule", "provided ocr content")
        for page in payload.get("pages") or []:
            if not isinstance(page, Mapping):
                continue
            page_number = int(page.get("page_number") or 0)
            for region in page.get("regions") or []:
                if not isinstance(region, Mapping):
                    continue
                content = _clean_text(region.get("content"))
                lowered = content.casefold()
                region_flags: list[QualityFlag] = []
                if region.get("status") == "failed":
                    region_flags.append(QualityFlag("failed", "OCR region status is failed", "error"))
                if sum(marker in lowered for marker in evaluator_markers) >= 2:
                    region_flags.append(QualityFlag("suspicious_completed", "contains OCR evaluator commentary", "error"))
                lines = [line.strip() for line in content.splitlines() if line.strip()]
                if lines and Counter(lines).most_common(1)[0][1] >= 8:
                    region_flags.append(QualityFlag("suspicious_completed", "contains degenerate repeated OCR lines", "error"))
                if region_flags:
                    findings[(page_number, int(region.get("index") or 0))] = tuple(region_flags)
        return findings

    def initial_flags(
        self,
        record: Mapping[str, Any],
        audit_flags: Sequence[QualityFlag] = (),
    ) -> tuple[QualityFlag, ...]:
        flags = list(audit_flags)
        status = str(record.get("status") or "missing")
        task_type = str(record.get("task_type") or "unknown")
        content = _clean_text(record.get("content"))
        if status == "failed" and not any(flag.code == "failed" for flag in flags):
            flags.append(QualityFlag("failed", "OCR region status is failed", "error"))
        if task_type != "figure" and not content and status not in {"failed", "preserved"}:
            flags.append(QualityFlag("empty_content", "textual region has no usable OCR content", "error"))
        if record.get("finish_reason") == "length":
            flags.append(QualityFlag("output_truncated", "model output stopped at the token limit", "error"))
        score = record.get("score")
        if isinstance(score, (int, float)) and float(score) < 0.3:
            flags.append(QualityFlag("low_detector_score", f"layout detector score is {float(score):.3f}", "warning"))
        if task_type == "table" and "<table" in content.casefold():
            if content.casefold().count("<table") != content.casefold().count("</table>"):
                flags.append(QualityFlag("malformed_html_table", "table HTML has unbalanced table tags", "error"))
        for token in ("<|image_pad|>", "<|vision_start|>", "<|vision_end|>"):
            if token in content:
                flags.append(QualityFlag("ocr_control_token", f"OCR content contains control token {token}", "error"))
        normalized_content = unicodedata.normalize("NFKC", content).upper()
        for raw_term, canonical_term in COMMON_IDENTIFIER_ALIASES.items():
            if raw_term != canonical_term and raw_term in normalized_content:
                flags.append(
                    QualityFlag(
                        "identifier_alias",
                        f"possible OCR identifier confusion {raw_term!r}; canonical alias is {canonical_term!r}",
                        "warning",
                    )
                )
        deduplicated: dict[tuple[str, str], QualityFlag] = {}
        for flag in flags:
            deduplicated[(flag.code, flag.reason)] = flag
        return tuple(deduplicated.values())

    @staticmethod
    def add_flag(region: LayoutRegion, flag: QualityFlag) -> None:
        if (flag.code, flag.reason) not in {(item.code, item.reason) for item in region.quality_flags}:
            region.quality_flags = (*region.quality_flags, flag)

    def resolve_native_recoverable_flags(self, region: LayoutRegion) -> None:
        resolution = "replaced canonical retrieval text with spatially aligned native PDF words"
        region.quality_flags = tuple(
            QualityFlag(
                code=flag.code,
                reason=flag.reason,
                severity=flag.severity,
                resolved=True,
                resolution=resolution,
            )
            if flag.code in self.native_recoverable_codes and flag.severity == "error"
            else flag
            for flag in region.quality_flags
        )

    def finalize(self, region: LayoutRegion) -> None:
        if region.task_type == "figure" and region.ocr_status == "preserved":
            region.quarantined = False
            return
        region.quarantined = any(
            flag.severity == "error" and not flag.resolved
            for flag in region.quality_flags
        )
        if region.task_type != "figure" and not region.canonical_text.strip():
            region.quarantined = True
            self.add_flag(region, QualityFlag("empty_canonical_text", "region has no canonical retrieval text", "error"))


class PDFTextReconciler:
    """Recover spatially aligned native PDF text while retaining every OCR alternative."""

    def __init__(
        self,
        *,
        agreement_threshold: float = 0.75,
        disagreement_threshold: float = 0.30,
        minimum_native_characters: int = 2,
    ):
        self.agreement_threshold = agreement_threshold
        self.disagreement_threshold = disagreement_threshold
        self.minimum_native_characters = minimum_native_characters

    @staticmethod
    def _extract_region_text(page: Any, region: LayoutRegion) -> str:
        normalized = region.bbox_normalized
        rectangle = page.rect
        clip = (
            rectangle.x0 + normalized[0] * rectangle.width,
            rectangle.y0 + normalized[1] * rectangle.height,
            rectangle.x0 + normalized[2] * rectangle.width,
            rectangle.y0 + normalized[3] * rectangle.height,
        )
        words = page.get_text("words", clip=clip, sort=True)
        if not words:
            return ""
        lines: dict[tuple[int, int], list[tuple[int, str]]] = defaultdict(list)
        for word in words:
            if len(word) >= 8 and str(word[4]).strip():
                lines[(int(word[5]), int(word[6]))].append((int(word[7]), str(word[4])))
        return "\n".join(
            " ".join(text for _, text in sorted(line_words))
            for _, line_words in sorted(lines.items())
        ).strip()

    def _usable_native_text(self, value: str) -> bool:
        compact = _normalized_compare_text(value)
        if len(compact) < self.minimum_native_characters:
            return False
        lines = [line.strip() for line in value.splitlines() if line.strip()]
        if lines and Counter(lines).most_common(1)[0][1] >= 8:
            return False
        lowered = value.casefold()
        return not ("ground truth image" in lowered and "according to rule" in lowered)

    def reconcile(self, document: LayoutDocument, quality_gate: OCRQualityGate) -> None:
        if document.paths.pdf_path is None:
            return
        try:
            import fitz
        except ImportError as error:
            raise RuntimeError("PyMuPDF is required for native PDF reconciliation") from error
        with fitz.open(document.paths.pdf_path) as pdf:
            if len(pdf) != len(document.pages):
                raise ValueError(f"PDF has {len(pdf)} pages but layout has {len(document.pages)}")
            for region in document.regions:
                if region.task_type == "figure":
                    continue
                native = self._extract_region_text(pdf[region.page_number - 1], region)
                region.native_text = native
                if not self._usable_native_text(native):
                    continue
                if region.task_type == "table" and len(_normalized_compare_text(native)) < 8:
                    continue
                severe = any(
                    flag.severity == "error" and not flag.resolved
                    for flag in region.quality_flags
                )
                ocr = region.ocr_content
                normalized_ocr = _normalized_compare_text(ocr)
                normalized_native = _normalized_compare_text(native)
                agreement = (
                    SequenceMatcher(None, normalized_ocr, normalized_native, autojunk=False).ratio()
                    if normalized_ocr and normalized_native
                    else 0.0
                )
                ocr_syntax = set(_syntax_signature(ocr)) - {"A", "/"}
                native_syntax = set(_syntax_signature(native)) - {"A", "/"}
                syntax_conflict = bool(
                    ocr_syntax
                    and native_syntax
                    and ocr_syntax != native_syntax
                )
                operator_conflict = bool(
                    (_operator_signature(ocr) or _operator_signature(native))
                    and _operator_signature(ocr) != _operator_signature(native)
                )
                trusted_native_correction = (
                    syntax_conflict
                    and not operator_conflict
                    and agreement >= 0.58
                )
                if severe or not normalized_ocr:
                    region.canonical_text = native
                    region.text_source = "native_pdf_recovery"
                    quality_gate.resolve_native_recoverable_flags(region)
                    quality_gate.add_flag(
                        region,
                        QualityFlag(
                            "native_pdf_recovery",
                            "canonical text recovered from spatially aligned native PDF words",
                            "info",
                        ),
                    )
                elif region.task_type == "table" and "<table" in ocr.casefold():
                    region.text_source = "ocr_html+native_pdf"
                elif trusted_native_correction:
                    region.canonical_text = native
                    region.text_source = "native_pdf_correction"
                    quality_gate.add_flag(
                        region,
                        QualityFlag(
                            "native_pdf_correction",
                            "spatially aligned native PDF corrected a high-agreement OCR identifier difference",
                            "info",
                        ),
                    )
                elif syntax_conflict or operator_conflict:
                    region.text_source = "ocr+native_pdf_conflict"
                    quality_gate.add_flag(
                        region,
                        QualityFlag(
                            "ocr_native_syntax_conflict",
                            "OCR and native PDF disagree on exact identifiers or meaning-bearing operators",
                            "error",
                        ),
                    )
                elif (
                    agreement >= self.agreement_threshold
                    and min(len(normalized_ocr), len(normalized_native))
                    / max(len(normalized_ocr), len(normalized_native))
                    >= 0.60
                ):
                    region.canonical_text = native
                    region.text_source = "native_pdf"
                else:
                    region.text_source = "ocr+native_pdf_alternative"
                    if agreement < self.disagreement_threshold:
                        quality_gate.add_flag(
                            region,
                            QualityFlag(
                                "ocr_native_disagreement",
                                f"normalized OCR/native agreement is {agreement:.3f}",
                                "warning",
                            ),
                        )


@dataclass(frozen=True)
class VisualAssetRecord:
    """A page or region image that can be handed to an external visual index."""

    asset_id: str
    document_id: str
    page_number: int
    asset_type: str
    path: Path
    parent_id: str
    region_ids: tuple[str, ...]
    bboxes_normalized: tuple[tuple[float, float, float, float], ...]
    indexable: bool = True


class PDFPageRenderer:
    """Materialize deterministic page images when OCR did not embed them."""

    def __init__(self, *, dpi: int = 150, image_format: str = "png"):
        if dpi <= 0:
            raise ValueError("render DPI must be positive")
        self.dpi = dpi
        self.image_format = image_format.casefold().lstrip(".")

    def render(
        self,
        document: LayoutDocument,
        output_dir: str | Path,
        *,
        page_numbers: Sequence[int] | None = None,
        overwrite: bool = False,
    ) -> tuple[VisualAssetRecord, ...]:
        if document.paths.pdf_path is None:
            raise ValueError("page rendering requires a source PDF")
        try:
            import fitz
        except ImportError as error:
            raise RuntimeError("PyMuPDF is required for page rendering") from error
        output = Path(output_dir)
        output.mkdir(parents=True, exist_ok=True)
        selected = set(page_numbers or range(1, len(document.pages) + 1))
        invalid = selected - set(range(1, len(document.pages) + 1))
        if invalid:
            raise ValueError(f"page numbers are outside the document: {sorted(invalid)}")
        digest = document.document_id.removeprefix("sha256:")[:16]
        records: list[VisualAssetRecord] = []
        with fitz.open(document.paths.pdf_path) as pdf:
            for page_number in sorted(selected):
                path = output / f"{digest}-page-{page_number:04d}.{self.image_format}"
                if overwrite or not path.is_file():
                    pixmap = pdf[page_number - 1].get_pixmap(dpi=self.dpi, alpha=False)
                    pixmap.save(path)
                page_regions = document.pages[page_number - 1].regions
                records.append(
                    VisualAssetRecord(
                        asset_id=f"{document.document_id}:page-image:{page_number:04d}:{self.dpi}dpi",
                        document_id=document.document_id,
                        page_number=page_number,
                        asset_type="page",
                        path=path,
                        parent_id=next((region.parent_id for region in page_regions if region.parent_id), ""),
                        region_ids=tuple(region.region_id for region in page_regions),
                        bboxes_normalized=tuple(region.bbox_normalized for region in page_regions),
                    )
                )
        return tuple(records)


def collect_region_visual_assets(document: LayoutDocument) -> tuple[VisualAssetRecord, ...]:
    """Expose table/figure/formula crops while suppressing nested figures as standalone hits."""
    records: list[VisualAssetRecord] = []
    for region in document.regions:
        if region.asset_path is None or region.task_type not in {"table", "figure", "formula"}:
            continue
        records.append(
            VisualAssetRecord(
                asset_id=f"{region.region_id}:crop",
                document_id=document.document_id,
                page_number=region.page_number,
                asset_type=region.task_type,
                path=region.asset_path,
                parent_id=region.parent_id,
                region_ids=(region.region_id,),
                bboxes_normalized=(region.bbox_normalized,),
                indexable=region.indexable and region.nested_parent_region_id is None,
            )
        )
    return tuple(records)


In [ ]:
# | export
class LayoutBundleLoader:
    """Load, validate, join, hash, quality-gate, and reconcile one evidence bundle."""

    def __init__(
        self,
        *,
        validator: LayoutBundleValidator | None = None,
        quality_gate: OCRQualityGate | None = None,
        reconciler: PDFTextReconciler | None = None,
        pipeline_version: str = LAYOUT_PIPELINE_VERSION,
    ):
        self.validator = validator or LayoutBundleValidator()
        self.quality_gate = quality_gate or OCRQualityGate()
        self.reconciler = reconciler or PDFTextReconciler()
        self.pipeline_version = pipeline_version

    @staticmethod
    def _document_metadata(payload: Mapping[str, Any], stem: str, regions: Sequence[LayoutRegion]) -> tuple[str, str, str]:
        title_region = next((region for region in regions if region.label == "doc_title" and region.ocr_content), None)
        bracketed_title = re.search(r"《([^》]+)》", stem)
        title = (
            bracketed_title.group(1).strip()
            if bracketed_title
            else _WHITESPACE_RE.sub(" ", title_region.ocr_content).strip()
            if title_region
            else stem
        )
        code_match = re.search(r"\b([A-Z]{2}\d{6})\b", stem)
        revision_match = re.search(r"\(([A-Z])[-/]?(\d+)\)", stem, flags=re.IGNORECASE)
        code = code_match.group(1) if code_match else ""
        revision = f"{revision_match.group(1).upper()}/{revision_match.group(2)}" if revision_match else ""
        return title, code, revision

    @staticmethod
    def _document_identity(payload: Mapping[str, Any], markdown_bytes: bytes, pdf_path: Path | None) -> tuple[str, dict[str, str]]:
        artifacts: dict[str, str] = {}
        if pdf_path is not None and pdf_path.is_file():
            artifacts["pdf"] = sha256_file(pdf_path)
            document_digest = artifacts["pdf"]
        else:
            projection = _compact_json(_stable_semantic_layout_projection(payload)).encode("utf-8")
            document_digest = _sha256_bytes(projection + b"\0" + markdown_bytes)
        return f"sha256:{document_digest}", artifacts

    def load(
        self,
        layout_path: str | Path,
        *,
        markdown_path: str | Path | None = None,
        pdf_path: str | Path | None = None,
        pdf_roots: Sequence[str | Path] = (),
        strict: bool = True,
        reconcile_native_text: bool = True,
    ) -> LayoutDocument:
        paths = LayoutBundlePaths.from_layout(
            layout_path,
            markdown_path=markdown_path,
            pdf_path=pdf_path,
            pdf_roots=pdf_roots,
        )
        validation_issues = self.validator.validate(paths, strict=strict)
        payload = json.loads(paths.layout_path.read_text(encoding="utf-8"))
        markdown_bytes = paths.markdown_path.read_bytes()
        markdown = markdown_bytes.decode("utf-8")
        markers = {marker.key: marker for marker in parse_markdown_regions(markdown)}
        document_id, artifact_hashes = self._document_identity(payload, markdown_bytes, paths.pdf_path)
        artifact_hashes["layout"] = sha256_file(paths.layout_path)
        artifact_hashes["markdown"] = _sha256_bytes(markdown_bytes)
        audit_flags = self.quality_gate.audit_flags(paths.layout_path)
        asset_prefix = payload.get("asset_prefix")

        pages: list[LayoutPage] = []
        all_regions: list[LayoutRegion] = []
        for raw_page in payload.get("pages") or []:
            page_number = int(raw_page["page_number"])
            width, height = int(raw_page["width"]), int(raw_page["height"])
            page_regions: list[LayoutRegion] = []
            for raw_region in raw_page.get("regions") or []:
                index = int(raw_region["index"])
                marker = markers[(page_number, index)]
                ocr_content = _clean_text(raw_region.get("content"))
                initial_text = ocr_content or _plain_markdown(marker.payload)
                quality_flags = self.quality_gate.initial_flags(
                    raw_region,
                    audit_flags.get((page_number, index), ()),
                )
                region = LayoutRegion(
                    document_id=document_id,
                    page_number=page_number,
                    page_width=width,
                    page_height=height,
                    region_index=index,
                    label=str(raw_region.get("label") or "unknown"),
                    detector_score=float(raw_region["score"]) if isinstance(raw_region.get("score"), (int, float)) else None,
                    bbox=BoundingBox.from_value(raw_region.get("bbox")) or marker.bbox,
                    task_type=str(raw_region.get("task_type") or "text"),
                    ocr_status=str(raw_region.get("status") or "missing"),
                    raw_content=_clean_text(raw_region.get("raw_content")),
                    ocr_content=ocr_content,
                    raw_markdown=f"{marker.marker}\n\n{marker.payload}".strip(),
                    asset_path=paths.resolve_asset(raw_region.get("asset"), prefix=asset_prefix),
                    provenance=dict(raw_region),
                    canonical_text=initial_text,
                    text_source="ocr" if ocr_content else "markdown_fallback",
                    quality_flags=quality_flags,
                )
                self.quality_gate.finalize(region)
                page_regions.append(region)
                all_regions.append(region)
            pages.append(
                LayoutPage(
                    page_number=page_number,
                    width=width,
                    height=height,
                    status=str(raw_page.get("status") or "missing"),
                    error=str(raw_page["error"]) if raw_page.get("error") else None,
                    regions=tuple(page_regions),
                )
            )

        for asset_path in sorted({region.asset_path for region in all_regions if region.asset_path is not None}):
            try:
                asset_key = asset_path.relative_to(paths.layout_path.parent).as_posix()
            except ValueError:
                asset_key = asset_path.name
            artifact_hashes[f"asset:{asset_key}"] = sha256_file(asset_path)

        for position, region in enumerate(all_regions):
            if position:
                region.previous_region_id = all_regions[position - 1].region_id
            if position + 1 < len(all_regions):
                region.next_region_id = all_regions[position + 1].region_id

        title, code, revision = self._document_metadata(payload, paths.stem, all_regions)
        settings = payload.get("settings") if isinstance(payload.get("settings"), Mapping) else {}
        document = LayoutDocument(
            document_id=document_id,
            document_title=title,
            document_code=code,
            revision=revision,
            schema_version=int(payload.get("schema_version") or 0),
            status=str(payload.get("status") or "missing"),
            paths=paths,
            artifact_hashes=artifact_hashes,
            provider=str(payload.get("provider") or settings.get("provider") or ""),
            ocr_model=str(payload.get("model") or settings.get("model") or ""),
            layout_model=str(payload.get("layout_model") or settings.get("layout_model") or ""),
            dpi=int(payload.get("dpi") or settings.get("dpi")) if payload.get("dpi") or settings.get("dpi") else None,
            pages=tuple(pages),
            raw_layout=payload,
            pipeline_version=self.pipeline_version,
            validation_issues=validation_issues,
        )
        if reconcile_native_text and paths.pdf_path is not None:
            self.reconciler.reconcile(document, self.quality_gate)
        for region in document.regions:
            self.quality_gate.finalize(region)
        return document

In [ ]:
# | export
def normalize_section_number(value: str) -> str:
    return ".".join(part.strip() for part in value.split("."))


def _instruction_code(title: str) -> str | None:
    cleaned = unicodedata.normalize("NFKC", title).strip().rstrip(":：")
    upper = _WHITESPACE_RE.sub(" ", cleaned).upper()
    for raw, canonical in sorted(MULTIWORD_INSTRUCTION_ALIASES.items(), key=lambda item: -len(item[0])):
        if upper == raw or upper.startswith(f"{raw} "):
            return canonical
    alias = COMMON_IDENTIFIER_ALIASES.get(cleaned.upper())
    if alias:
        return alias
    match = _IDENTIFIER_RE.search(cleaned)
    return COMMON_IDENTIFIER_ALIASES.get(match.group(0).upper(), match.group(0)) if match else None


class LayoutHierarchyBuilder:
    """Infer numbered section/instruction parents while preserving region order."""

    boilerplate_labels = frozenset({"header", "footer", "number"})

    def __init__(self, *, repeated_page_ratio: float = 0.20):
        self.repeated_page_ratio = repeated_page_ratio

    @staticmethod
    def _heading(region: LayoutRegion) -> tuple[str, str] | None:
        if region.label not in {"paragraph_title", "title"}:
            return None
        for candidate in (region.canonical_text, region.native_text, region.ocr_content):
            collapsed = _WHITESPACE_RE.sub(" ", _plain_markdown(candidate)).strip()
            match = _NUMBERED_HEADING_RE.match(collapsed)
            if match:
                return normalize_section_number(match.group("number")), _canonical_heading_title(match.group("title"))
        return None

    def _mark_boilerplate(self, document: LayoutDocument) -> None:
        by_text_pages: dict[str, set[int]] = defaultdict(set)
        for region in document.regions:
            normalized = _normalized_compare_text(region.canonical_text)
            if normalized:
                by_text_pages[normalized].add(region.page_number)
        threshold = max(3, round(len(document.pages) * self.repeated_page_ratio))
        repeated = {text for text, pages in by_text_pages.items() if len(pages) >= threshold}
        for region in document.regions:
            normalized = _normalized_compare_text(region.canonical_text)
            region.is_boilerplate = region.label in self.boilerplate_labels or bool(normalized and normalized in repeated)

    @staticmethod
    def _summary_text(region: LayoutRegion) -> str:
        text = region.canonical_text.strip()
        if not text:
            return ""
        if region.task_type == "table":
            return normalize_html_table(text).text
        if "<" in text and ">" in text:
            text = re.sub(r"<[^>]+>", " ", html.unescape(text))
        return _WHITESPACE_RE.sub(" ", text).strip()

    @staticmethod
    def _bounded_summary(units: Sequence[str], limit: int = 4000) -> str:
        selected: list[str] = []
        used = 0
        for unit in units:
            if not unit:
                continue
            separator = 1 if selected else 0
            if used + separator + len(unit) <= limit:
                selected.append(unit)
                used += separator + len(unit)
                continue
            remaining = limit - used - separator
            if remaining >= 80:
                boundary = max(
                    unit.rfind("。", 0, remaining),
                    unit.rfind("；", 0, remaining),
                    unit.rfind("\n", 0, remaining),
                    unit.rfind(" ", 0, remaining),
                )
                selected.append(unit[: boundary + 1 if boundary >= remaining // 2 else remaining].rstrip())
            break
        return "\n".join(selected)

    @staticmethod
    def _mark_nested_figures(document: LayoutDocument) -> None:
        for page in document.pages:
            tables = [region for region in page.regions if region.task_type == "table"]
            for figure in (region for region in page.regions if region.task_type == "figure"):
                parent = next((table for table in tables if table.bbox.contains(figure.bbox, tolerance=4)), None)
                if parent is not None:
                    figure.nested_parent_region_id = parent.region_id

    def build(self, document: LayoutDocument) -> tuple[LayoutSection, ...]:
        self._mark_boilerplate(document)
        self._mark_nested_figures(document)
        root = LayoutSection(
            parent_id=f"{document.document_id}:section:root",
            document_id=document.document_id,
            number=None,
            title=document.document_title,
            level=0,
            section_path=document.document_title,
            parent_section_id=None,
            page_start=1,
            page_end=len(document.pages),
        )
        sections: list[LayoutSection] = [root]
        sections_by_id: dict[str, LayoutSection] = {root.parent_id: root}
        active: dict[int, LayoutSection] = {0: root}
        toc_section: LayoutSection | None = None
        current = root
        for region in document.regions:
            plain = _plain_markdown(region.canonical_text).strip()
            if region.is_boilerplate:
                region.parent_id = current.parent_id
                region.section_path = current.section_path
                continue
            if plain in {"目录", "目 录"}:
                toc_section = LayoutSection(
                    parent_id=f"{document.document_id}:section:toc",
                    document_id=document.document_id,
                    number=None,
                    title="目录",
                    level=1,
                    section_path="目录",
                    parent_section_id=root.parent_id,
                    page_start=region.page_number,
                    page_end=region.page_number,
                    is_toc=True,
                )
                sections.append(toc_section)
                sections_by_id[toc_section.parent_id] = toc_section
                current = toc_section
            heading = self._heading(region)
            if heading is not None:
                number, title = heading
                level = number.count(".") + 1
                deepest_level = max(active)
                if (
                    level in active
                    and active[level].number == number
                    and deepest_level > level
                    and title != active[level].title
                    and _instruction_code(title)
                ):
                    previous = active[deepest_level].number or ""
                    previous_parts = previous.split(".")
                    if previous_parts[:-1] == number.split("."):
                        number = ".".join((*number.split("."), str(int(previous_parts[-1]) + 1)))
                        level += 1
                        self._append_region_flag(
                            region,
                            QualityFlag("inferred_section_number", f"inferred section number {number} from document sequence", "warning"),
                        )
                for stale_level in [item for item in active if item >= level]:
                    active.pop(stale_level, None)
                number_parts = number.split(".")
                for ancestor_level in range(1, level):
                    expected_number = ".".join(number_parts[:ancestor_level])
                    if active.get(ancestor_level) is not None and active[ancestor_level].number == expected_number:
                        continue
                    for stale_level in [item for item in active if item >= ancestor_level]:
                        active.pop(stale_level, None)
                    ancestor_id = f"{document.document_id}:section:{expected_number}"
                    ancestor = sections_by_id.get(ancestor_id)
                    if ancestor is None:
                        ancestor_parent = active.get(ancestor_level - 1, root)
                        ancestor_path = " > ".join(
                            [
                                *(
                                    section.title if section.number is None else f"{section.number} {section.title}"
                                    for depth, section in sorted(active.items())
                                    if depth and depth < ancestor_level
                                ),
                                expected_number,
                            ]
                        )
                        ancestor = LayoutSection(
                            parent_id=ancestor_id,
                            document_id=document.document_id,
                            number=expected_number,
                            title="",
                            level=ancestor_level,
                            section_path=ancestor_path,
                            parent_section_id=ancestor_parent.parent_id,
                            page_start=region.page_number,
                            page_end=region.page_number,
                        )
                        sections.append(ancestor)
                        sections_by_id[ancestor_id] = ancestor
                    active[ancestor_level] = ancestor
                parent = active.get(level - 1, root)
                path_parts = [section for depth, section in sorted(active.items()) if depth and depth < level]
                title_with_number = f"{number} {title}".strip()
                section_path = " > ".join([*(section.title if section.number is None else f"{section.number} {section.title}" for section in path_parts), title_with_number])
                parent_id = f"{document.document_id}:section:{number}"
                current = sections_by_id.get(parent_id)
                if current is None:
                    current = LayoutSection(
                        parent_id=parent_id,
                        document_id=document.document_id,
                        number=number,
                        title=title,
                        level=level,
                        section_path=section_path,
                        parent_section_id=parent.parent_id,
                        page_start=region.page_number,
                        page_end=region.page_number,
                        instruction_code=_instruction_code(title) if level >= 3 else None,
                    )
                    sections.append(current)
                    sections_by_id[parent_id] = current
                elif not current.title:
                    current.title = title
                    current.section_path = section_path
                    current.instruction_code = _instruction_code(title) if level >= 3 else None
                    current.page_start = min(current.page_start, region.page_number)
                elif current.title != title:
                    self._append_region_flag(
                        region,
                        QualityFlag(
                            "duplicate_section_number",
                            f"section number {number} repeats with title {title!r}; retained first title {current.title!r}",
                            "warning",
                        ),
                    )
                active[level] = current
                toc_section = None
            elif toc_section is not None:
                current = toc_section
                region.is_toc = True

            region.parent_id = current.parent_id
            region.section_number = current.number
            region.section_path = current.section_path
            region.instruction_code = current.instruction_code
            region.is_toc = current.is_toc
            current.region_ids.append(region.region_id)
            current.page_end = max(current.page_end, region.page_number)

        region_map = document.region_map
        for section in sections:
            texts = [
                self._summary_text(region_map[region_id])
                for region_id in section.region_ids
                if region_id in region_map
                and region_map[region_id].canonical_text.strip()
                and not region_map[region_id].quarantined
                and not region_map[region_id].is_boilerplate
            ]
            section.summary = self._bounded_summary(texts)
        document.sections = tuple(sections)
        return document.sections

    @staticmethod
    def _append_region_flag(region: LayoutRegion, flag: QualityFlag) -> None:
        if (flag.code, flag.reason) not in {(item.code, item.reason) for item in region.quality_flags}:
            region.quality_flags = (*region.quality_flags, flag)


@dataclass(frozen=True)
class NormalizedTable:
    original: str
    text: str
    rows: tuple[tuple[str, ...], ...]


class _TableHTMLParser(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.tables: list[list[list[tuple[str, int, int]]]] = []
        self._table_depth = 0
        self._rows: list[list[tuple[str, int, int]]] = []
        self._row: list[tuple[str, int, int]] | None = None
        self._cell_parts: list[str] | None = None
        self._rowspan = 1
        self._colspan = 1
        self.outside_parts: list[str] = []

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        tag = tag.casefold()
        if tag == "table":
            if self._table_depth == 0:
                self._rows = []
            self._table_depth += 1
        elif tag == "tr" and self._table_depth:
            self._row = []
        elif tag in {"td", "th"} and self._table_depth and self._row is not None:
            attributes = dict(attrs)
            try:
                self._rowspan = max(1, int(attributes.get("rowspan") or 1))
            except (TypeError, ValueError):
                self._rowspan = 1
            try:
                self._colspan = max(1, int(attributes.get("colspan") or 1))
            except (TypeError, ValueError):
                self._colspan = 1
            self._cell_parts = []
        elif tag == "br" and self._cell_parts is not None:
            self._cell_parts.append("\n")

    def handle_data(self, data: str) -> None:
        if self._cell_parts is not None:
            self._cell_parts.append(data)
        elif self._table_depth == 0 and data.strip():
            self.outside_parts.append(data.strip())

    def handle_endtag(self, tag: str) -> None:
        tag = tag.casefold()
        if tag in {"td", "th"} and self._cell_parts is not None and self._row is not None:
            value = _WHITESPACE_RE.sub(" ", "".join(self._cell_parts)).strip()
            self._row.append((value, self._rowspan, self._colspan))
            self._cell_parts = None
        elif tag == "tr" and self._row is not None:
            self._rows.append(self._row)
            self._row = None
        elif tag == "table" and self._table_depth:
            self._table_depth -= 1
            if self._table_depth == 0:
                self.tables.append(self._rows)
                self._rows = []


def _expand_table_rows(rows: Sequence[Sequence[tuple[str, int, int]]]) -> tuple[tuple[str, ...], ...]:
    grid: list[list[str | None]] = []
    for row_index, row in enumerate(rows):
        while len(grid) <= row_index:
            grid.append([])
        column = 0
        for value, rowspan, colspan in row:
            while column < len(grid[row_index]) and grid[row_index][column] is not None:
                column += 1
            for target_row in range(row_index, row_index + rowspan):
                while len(grid) <= target_row:
                    grid.append([])
                needed = column + colspan
                if len(grid[target_row]) < needed:
                    grid[target_row].extend([None] * (needed - len(grid[target_row])))
                for target_column in range(column, column + colspan):
                    grid[target_row][target_column] = value
            column += colspan
    width = max((len(row) for row in grid), default=0)
    return tuple(tuple((cell or "") for cell in [*row, *([None] * (width - len(row)))]) for row in grid)


def normalize_html_table(value: str) -> NormalizedTable:
    """Expand HTML row/column spans while preserving literal parameter tokens."""
    if "<table" not in value.casefold():
        text = _WHITESPACE_RE.sub(" ", html.unescape(value)).strip()
        return NormalizedTable(original=value, text=text, rows=((text,),) if text else ())
    parser = _TableHTMLParser()
    parser.feed(value)
    all_rows: list[tuple[str, ...]] = []
    for table in parser.tables:
        all_rows.extend(_expand_table_rows(table))
    lines = [" | ".join(cell for cell in row if cell).strip() for row in all_rows]
    outside = " ".join(parser.outside_parts).strip()
    text = "\n".join([*( [outside] if outside else []), *(line for line in lines if line)])
    return NormalizedTable(original=value, text=text, rows=tuple(all_rows))


class LayoutChunker:
    """Create typed, citation-ready children without crossing instruction parents."""

    def __init__(
        self,
        *,
        max_prose_characters: int = 2400,
        max_table_characters: int = 2600,
        max_table_rows: int = 12,
    ):
        self.max_prose_characters = max_prose_characters
        self.max_table_characters = max_table_characters
        self.max_table_rows = max_table_rows

    @staticmethod
    def _aliases(regions: Sequence[LayoutRegion], instruction_code: str | None) -> tuple[str, ...]:
        aliases: set[str] = set()
        joined = "\n".join(region.canonical_text for region in regions)
        joined_upper = joined.upper()
        for raw, canonical in COMMON_IDENTIFIER_ALIASES.items():
            if raw in joined_upper or canonical in joined_upper:
                aliases.update({raw, canonical})
        if instruction_code:
            aliases.add(instruction_code)
        aliases.update(_IDENTIFIER_RE.findall(joined))
        return tuple(sorted(aliases))

    @staticmethod
    def _record_type_label(content_type: str) -> str:
        return {
            "prose": "正文",
            "table_rows": "表格与参数",
            "figure": "图示",
            "formula": "公式",
            "toc": "目录",
        }.get(content_type, content_type)

    @staticmethod
    def _source_hashes(
        document: LayoutDocument,
        regions: Sequence[LayoutRegion],
    ) -> tuple[tuple[str, str], ...]:
        keys = {key for key in ("pdf", "layout", "markdown") if key in document.artifact_hashes}
        for region in regions:
            if region.asset_path is None:
                continue
            try:
                relative = region.asset_path.relative_to(document.paths.layout_path.parent).as_posix()
            except ValueError:
                relative = region.asset_path.name
            key = f"asset:{relative}"
            if key in document.artifact_hashes:
                keys.add(key)
        return tuple((key, document.artifact_hashes[key]) for key in sorted(keys))

    def _make_record(
        self,
        document: LayoutDocument,
        regions: Sequence[LayoutRegion],
        *,
        exact_text: str,
        content_type: str,
        sequence: int,
    ) -> LayoutRetrievalRecord:
        first, last = regions[0], regions[-1]
        instruction_code = first.instruction_code
        aliases = self._aliases(regions, instruction_code)
        keyword_text = ", ".join(aliases)
        retrieval_text = "\n".join(
            item
            for item in (
                f"文档: {document.document_title} {document.revision}".strip(),
                f"章节: {first.section_path}" if first.section_path else "",
                f"类型: {self._record_type_label(content_type)}",
                f"关键词: {keyword_text}" if keyword_text else "",
                f"正文: {exact_text.strip()}",
            )
            if item
        )
        span = f"p{first.page_number:04d}r{first.region_index:04d}-p{last.page_number:04d}r{last.region_index:04d}"
        version_tag = _sha256_bytes(document.pipeline_version.encode("utf-8"))[:10]
        identity_digest = _sha256_bytes(
            "\0".join(
                (
                    document.pipeline_version,
                    first.parent_id,
                    content_type,
                    *(region.region_id for region in regions),
                    exact_text.strip(),
                )
            ).encode("utf-8")
        )[:20]
        chunk_id = f"{document.document_id}:{span}:v{version_tag}:h{identity_digest}"
        quality_flags = tuple(dict.fromkeys(flag.code for region in regions for flag in region.quality_flags))
        quality_reasons = tuple(dict.fromkeys(flag.reason for region in regions for flag in region.quality_flags))
        asset_paths = tuple(dict.fromkeys(str(region.asset_path) for region in regions if region.asset_path is not None))
        return LayoutRetrievalRecord(
            chunk_id=chunk_id,
            parent_id=first.parent_id,
            document_id=document.document_id,
            document_title=document.document_title,
            document_code=document.document_code,
            revision=document.revision,
            section_path=first.section_path,
            instruction_code=instruction_code,
            content_type=content_type,
            retrieval_text=retrieval_text,
            exact_text=exact_text.strip(),
            raw_markdown="\n\n".join(region.raw_markdown for region in regions),
            raw_ocr="\n\n".join(region.raw_content for region in regions),
            canonical_texts=tuple(region.canonical_text for region in regions),
            cleaned_ocr_texts=tuple(region.ocr_content for region in regions),
            native_texts=tuple(region.native_text for region in regions),
            detector_scores=tuple(region.detector_score for region in regions),
            page_start=first.page_number,
            page_end=last.page_number,
            region_ids=tuple(region.region_id for region in regions),
            bboxes=tuple(region.bbox.as_tuple() for region in regions),
            bboxes_normalized=tuple(region.bbox_normalized for region in regions),
            region_labels=tuple(region.label for region in regions),
            ocr_statuses=tuple(region.ocr_status for region in regions),
            quality_flags=quality_flags,
            quality_reasons=quality_reasons,
            region_quality_flags=tuple(
                tuple(asdict(flag) for flag in region.quality_flags)
                for region in regions
            ),
            text_sources=tuple(region.text_source for region in regions),
            asset_paths=asset_paths,
            region_asset_paths=tuple(
                str(region.asset_path) if region.asset_path is not None else None
                for region in regions
            ),
            source_pdf_uri=str(document.paths.pdf_path) if document.paths.pdf_path is not None else None,
            source_markdown_uri=str(document.paths.markdown_path),
            source_layout_uri=str(document.paths.layout_path),
            source_hashes=self._source_hashes(document, regions),
            content_sha256=_sha256_bytes(exact_text.strip().encode("utf-8")),
            pipeline_version=document.pipeline_version,
            aliases=aliases,
            indexable=all(region.indexable for region in regions)
            and not any(region.is_toc for region in regions)
            and not any(region.nested_parent_region_id for region in regions if region.task_type == "figure"),
        )

    @staticmethod
    def _split_text(value: str, limit: int) -> tuple[str, ...]:
        if len(value) <= limit:
            return (value,)
        units = [unit.strip() for unit in re.split(r"(?<=[。！？；.!?;])\s*|\n+", value) if unit.strip()]
        if len(units) <= 1:
            chunks: list[str] = []
            remaining = value.strip()
            while len(remaining) > limit:
                floor = max(1, limit // 2)
                boundary = max(
                    remaining.rfind(" ", floor, limit + 1),
                    remaining.rfind("，", floor, limit + 1),
                    remaining.rfind(",", floor, limit + 1),
                    remaining.rfind("/", floor, limit + 1),
                )
                cut = boundary + 1 if boundary >= floor else limit
                chunks.append(remaining[:cut].rstrip())
                remaining = remaining[cut:].lstrip()
            if remaining:
                chunks.append(remaining)
            return tuple(chunks)
        chunks: list[str] = []
        current = ""
        for unit in units:
            if len(unit) > limit:
                if current:
                    chunks.append(current)
                    current = ""
                chunks.extend(unit[start : start + limit] for start in range(0, len(unit), limit))
            elif current and len(current) + 1 + len(unit) > limit:
                chunks.append(current)
                current = unit
            else:
                current = f"{current}\n{unit}".strip()
        if current:
            chunks.append(current)
        return tuple(chunks)

    def _table_records(
        self,
        document: LayoutDocument,
        region: LayoutRegion,
        sequence: int,
    ) -> tuple[list[LayoutRetrievalRecord], int]:
        normalized = normalize_html_table(region.canonical_text)
        if not normalized.rows or len(normalized.text) <= self.max_table_characters:
            return [self._make_record(document, [region], exact_text=normalized.text, content_type="toc" if region.is_toc else "table_rows", sequence=sequence)], sequence + 1
        if len(normalized.rows) == 1:
            records = [
                self._make_record(
                    document,
                    [region],
                    exact_text=part,
                    content_type="toc" if region.is_toc else "table_rows",
                    sequence=sequence + offset,
                )
                for offset, part in enumerate(self._split_text(normalized.text, self.max_table_characters))
            ]
            return records, sequence + len(records)
        header = normalized.rows[0]
        groups: list[list[tuple[str, ...]]] = []
        current: list[tuple[str, ...]] = []
        current_size = 0
        for row in normalized.rows[1:]:
            row_text = " | ".join(cell for cell in row if cell)
            if current and (len(current) >= self.max_table_rows or current_size + len(row_text) > self.max_table_characters):
                groups.append(current)
                current, current_size = [], 0
            current.append(row)
            current_size += len(row_text)
        if current:
            groups.append(current)
        records: list[LayoutRetrievalRecord] = []
        for group in groups:
            rows = [header, *group]
            text = "\n".join(" | ".join(cell for cell in row if cell) for row in rows)
            records.append(self._make_record(document, [region], exact_text=text, content_type="toc" if region.is_toc else "table_rows", sequence=sequence))
            sequence += 1
        return records, sequence

    @staticmethod
    def _figure_text(region: LayoutRegion, regions: Sequence[LayoutRegion], position: int) -> str:
        candidates = []
        for offset in (1, -1, 2, -2):
            candidate_position = position + offset
            if 0 <= candidate_position < len(regions):
                candidate = regions[candidate_position]
                if candidate.page_number == region.page_number and candidate.label == "figure_title":
                    candidates.append(candidate.canonical_text)
                    break
        return candidates[0] if candidates else (region.section_path or "图示")

    def chunk(self, document: LayoutDocument) -> tuple[LayoutRetrievalRecord, ...]:
        if not document.sections:
            LayoutHierarchyBuilder().build(document)
        records: list[LayoutRetrievalRecord] = []
        prose_group: list[LayoutRegion] = []
        prose_size = 0
        sequence = 1
        regions = list(document.regions)

        def flush_prose() -> None:
            nonlocal prose_group, prose_size, sequence
            if not prose_group:
                return
            exact_text = "\n\n".join(region.canonical_text for region in prose_group if region.canonical_text.strip())
            if exact_text:
                content_type = "toc" if prose_group[0].is_toc else "prose"
                records.append(self._make_record(document, prose_group, exact_text=exact_text, content_type=content_type, sequence=sequence))
                sequence += 1
            prose_group, prose_size = [], 0

        for position, region in enumerate(regions):
            if region.is_boilerplate:
                continue
            if region.task_type == "table":
                flush_prose()
                table_records, sequence = self._table_records(document, region, sequence)
                records.extend(table_records)
                continue
            if region.task_type == "figure":
                flush_prose()
                records.append(
                    self._make_record(
                        document,
                        [region],
                        exact_text=self._figure_text(region, regions, position),
                        content_type="figure",
                        sequence=sequence,
                    )
                )
                sequence += 1
                continue
            if region.task_type == "formula":
                flush_prose()
                if region.canonical_text.strip():
                    records.append(
                        self._make_record(
                            document,
                            [region],
                            exact_text=region.canonical_text,
                            content_type="formula",
                            sequence=sequence,
                        )
                    )
                    sequence += 1
                continue
            text = region.canonical_text.strip()
            if not text:
                flush_prose()
                continue
            boundary_changed = bool(prose_group) and (
                prose_group[-1].parent_id != region.parent_id
                or prose_group[-1].is_toc != region.is_toc
                or prose_group[-1].quarantined != region.quarantined
            )
            if boundary_changed or (prose_group and prose_size + len(text) > self.max_prose_characters):
                flush_prose()
            if len(text) > self.max_prose_characters:
                for part in self._split_text(text, self.max_prose_characters):
                    content_type = "toc" if region.is_toc else "prose"
                    records.append(
                        self._make_record(
                            document,
                            [region],
                            exact_text=part,
                            content_type=content_type,
                            sequence=sequence,
                        )
                    )
                    sequence += 1
                continue
            prose_group.append(region)
            prose_size += len(text)
        flush_prose()
        duplicate_ids: dict[str, int] = defaultdict(int)
        for record in records:
            duplicate_ids[record.chunk_id] += 1
            if duplicate_ids[record.chunk_id] > 1:
                record.chunk_id = f"{record.chunk_id}:d{duplicate_ids[record.chunk_id]}"
        for order, record in enumerate(records):
            record.reading_order = order
        return tuple(records)

In [ ]:
# | export
def ingest_layout_bundle(
    layout_path: str | Path,
    *,
    markdown_path: str | Path | None = None,
    pdf_path: str | Path | None = None,
    pdf_roots: Sequence[str | Path] = (),
    pipeline_version: str = LAYOUT_PIPELINE_VERSION,
    reconcile_native_text: bool = True,
    strict: bool = True,
    loader: LayoutBundleLoader | None = None,
    hierarchy_builder: LayoutHierarchyBuilder | None = None,
    chunker: LayoutChunker | None = None,
) -> LayoutIngestionResult:
    """Run canonical ingestion through typed retrieval-record construction."""
    bundle_loader = loader or LayoutBundleLoader(pipeline_version=pipeline_version)
    document = bundle_loader.load(
        layout_path,
        markdown_path=markdown_path,
        pdf_path=pdf_path,
        pdf_roots=pdf_roots,
        strict=strict,
        reconcile_native_text=reconcile_native_text,
    )
    sections = (hierarchy_builder or LayoutHierarchyBuilder()).build(document)
    records = (chunker or LayoutChunker()).chunk(document)
    return LayoutIngestionResult(document=document, sections=sections, records=records)

In [ ]:
# | hide
# | notest
import nbdev

nbdev.nbdev_export()